In [2]:
import requests
import pandas as pd
from pathlib import Path

In [3]:
cities = {
    "Karachi": (24.8607, 67.0011),
    "Peshawar": (34.0151, 71.5249),
    "Quetta": (30.1798, 66.9750),
    "Islamabad": (33.6844, 73.0479),
    "Lahore": (31.5204, 74.3587)
}

In [4]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "Asia/Karachi"
}

In [5]:
weather_dfs = {}

for city, (lat, lon) in cities.items():

    city_params = params.copy()

    city_params["latitude"] = lat
    city_params["longitude"] = lon

    response = requests.get(
        url,
        params=city_params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    city_df = pd.DataFrame(data["daily"])

    city_df["city"] = city

    weather_dfs[city] = city_df

    print(f"{city}: Data collected successfully")

Karachi: Data collected successfully
Peshawar: Data collected successfully
Quetta: Data collected successfully
Islamabad: Data collected successfully
Lahore: Data collected successfully


In [6]:
combined_df = pd.concat(
    weather_dfs.values(),
    ignore_index=True
)

combined_df.head()

,time,temperature_2m_max,temperature_2m_min,precipitation_sum,city
0,2025-01-01,26.2,15.2,0.0,Karachi
1,2025-01-02,25.6,14.8,0.0,Karachi
2,2025-01-03,24.6,14.8,0.0,Karachi
3,2025-01-04,24.2,17.2,0.0,Karachi
4,2025-01-05,22.5,13.0,0.0,Karachi


In [7]:
combined_df.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "precipitation"
}, inplace=True)

In [8]:
combined_df = combined_df[
    ["date", "city", "temp_max", "temp_min", "precipitation"]
]

combined_df.head()

,date,city,temp_max,temp_min,precipitation
0,2025-01-01,Karachi,26.2,15.2,0.0
1,2025-01-02,Karachi,25.6,14.8,0.0
2,2025-01-03,Karachi,24.6,14.8,0.0
3,2025-01-04,Karachi,24.2,17.2,0.0
4,2025-01-05,Karachi,22.5,13.0,0.0


In [9]:
# Data cleaning

combined_df["date"] = pd.to_datetime(combined_df["date"])

combined_df["temp_max"] = pd.to_numeric(
    combined_df["temp_max"], errors="coerce"
)

combined_df["temp_min"] = pd.to_numeric(
    combined_df["temp_min"], errors="coerce"
)

combined_df["precipitation"] = pd.to_numeric(
    combined_df["precipitation"], errors="coerce"
)

In [10]:
combined_df.isnull().sum()

date             0
city             0
temp_max         0
temp_min         0
precipitation    0
dtype: int64

In [11]:
combined_df.duplicated().sum()

np.int64(0)

In [12]:
combined_df["city"].value_counts()

city
Karachi      365
Peshawar     365
Quetta       365
Islamabad    365
Lahore       365
Name: count, dtype: int64

In [13]:
combined_df.dtypes

date             datetime64[us]
city                        str
temp_max                float64
temp_min                float64
precipitation           float64
dtype: object

In [14]:
combined_df.shape

(1825, 5)

In [15]:
output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)

combined_df.to_csv(
    output_path / "pakistan_weather_2025.csv",
    index=False
)

In [16]:
df = pd.read_csv(
    output_path / "pakistan_weather_2025.csv"
)

df.head()

,date,city,temp_max,temp_min,precipitation
0,2025-01-01,Karachi,26.2,15.2,0.0
1,2025-01-02,Karachi,25.6,14.8,0.0
2,2025-01-03,Karachi,24.6,14.8,0.0
3,2025-01-04,Karachi,24.2,17.2,0.0
4,2025-01-05,Karachi,22.5,13.0,0.0
